# Parte A — Pretraining MAE-AST su FSD50K
### Ablation sul masking ratio: 25% vs 50% vs 75%

**Obiettivo**: verificare che accuracy(75%) > accuracy(50%) > accuracy(25%) su ESC-50.

**Hardware**: A100 (Colab Pro). **Tempo stimato**: ~2.5 ore totali.

**Comando usato**: `fairseq-hydra-train` — esattamente quello del README MAE-AST.

**Note sul config** (verificate dal file `config/pretrain/mae_ast.yaml` degli autori):
- Task: `mae_ast_pretraining` — calcola le fbank features internamente
- Input TSV: percorsi assoluti a file .flac, lunghezza in campioni audio grezzi
- min_sample_size: 32000 (2s), max_sample_size: 250000 (15.6s)

In [ ]:
# CELLA 0 — Verifica GPU e spazio disco
import torch, shutil

if not torch.cuda.is_available():
    raise SystemExit('GPU non trovata. Runtime -> Change runtime type -> A100')

nome_gpu = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU:   {nome_gpu}  ({vram_gb:.1f} GB VRAM)')

if 'A100' not in nome_gpu:
    print(f'ATTENZIONE: GPU e {nome_gpu}, non A100. Cambiarla in Runtime -> Change runtime type')

totale, usato, libero = shutil.disk_usage('/')
print(f'Disco: {libero/1e9:.0f} GB liberi su {totale/1e9:.0f} GB')
if libero/1e9 < 50:
    print('ATTENZIONE: servono almeno 50 GB liberi per FSD50K')

In [ ]:
# CELLA 1 — Installa fairseq e dipendenze
#
# Usiamo la versione fairseq da PyPI.
# Il codice MAE-AST usa fairseq come libreria base —
# il repo degli autori aggiunge i componenti custom
# (model, criterion, task) via --user-dir.
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install',
    'fairseq', 'soundfile', 'librosa==0.10.1',
    'timm==0.9.16', '--quiet'], check=True)

import fairseq
print(f'fairseq: {fairseq.__version__}')

# Verifica che fairseq-hydra-train sia disponibile
result = subprocess.run(['which', 'fairseq-hydra-train'], capture_output=True, text=True)
if result.returncode == 0:
    print(f'fairseq-hydra-train: {result.stdout.strip()}')
else:
    print('ATTENZIONE: fairseq-hydra-train non trovato nel PATH')
    print('Provo con: python -m fairseq_cli.hydra_train')

In [ ]:
# CELLA 2 — Clona repo progetto e repo MAE-AST degli autori
import os, sys

# Repo nostro progetto
REPO_URL = 'https://github.com/alessiopgg/deep_learning_project.git'
REPO_DIR = '/content/mae-ast-poggesi'
if os.path.exists(REPO_DIR):
    os.system(f'cd {REPO_DIR} && git pull --quiet')
else:
    os.system(f'git clone {REPO_URL} {REPO_DIR} --quiet')
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(f'Repo progetto: {REPO_DIR}')

# Repo MAE-AST degli autori
# Struttura rilevante:
#   mae_ast/models/mae_ast.py      - architettura
#   mae_ast/criterions/mae_ast.py  - loss (MSE + InfoNCE)
#   mae_ast/tasks/mae_ast_pretraining.py - task (legge TSV, calcola fbank)
#   config/pretrain/mae_ast.yaml   - config Hydra di default
MAE_AST_DIR = '/content/MAE-AST-Public'
CONFIG_DIR  = f'{MAE_AST_DIR}/config/pretrain'
USER_DIR    = f'{MAE_AST_DIR}/mae_ast'

if not os.path.exists(MAE_AST_DIR):
    print('Clono MAE-AST-Public...')
    os.system(f'git clone https://github.com/AlanBaade/MAE-AST-Public.git {MAE_AST_DIR} --quiet')

print(f'Repo MAE-AST: {MAE_AST_DIR}')
for path in [USER_DIR, CONFIG_DIR,
             f'{USER_DIR}/models', f'{USER_DIR}/criterions', f'{USER_DIR}/tasks',
             f'{CONFIG_DIR}/mae_ast.yaml']:
    print(f'  {"OK" if os.path.exists(path) else "MANCANTE"}  {path}')

In [ ]:
# CELLA 3 — Monta Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_DIR      = '/content/drive/MyDrive/mae_ast_partA'
TSV_DIR_DRIVE  = f'{DRIVE_DIR}/fsd50k_tsv'
CKPT_DIR_DRIVE = f'{DRIVE_DIR}/checkpoints'

os.makedirs(TSV_DIR_DRIVE,  exist_ok=True)
os.makedirs(CKPT_DIR_DRIVE, exist_ok=True)

print(f'Drive montato')
print(f'TSV:          {TSV_DIR_DRIVE}')
print(f'Checkpoints:  {CKPT_DIR_DRIVE}')

In [ ]:
# CELLA 4 — Scarica FSD50K, converti in flac, genera TSV
#
# Il task mae_ast_pretraining si aspetta:
#   - File audio .flac (non .wav)
#   - TSV con prima riga "/" e percorsi assoluti
#   - Lunghezze in campioni audio grezzi a 16kHz
#     (il task calcola fbank internamente)
#
# Se i TSV sono gia su Drive li usiamo, MA dobbiamo
# verificare che i file audio esistano in questa sessione
# (i percorsi nel TSV sono assoluti a /content/FSD50K)

import os, shutil

TSV_DIR_LOCALE = '/content/fsd50k_tsv'
AUDIO_DIR      = '/content/FSD50K'
TRAIN_TSV      = f'{TSV_DIR_LOCALE}/train.tsv'
VALID_TSV      = f'{TSV_DIR_LOCALE}/valid.tsv'

def tsv_audio_exists(tsv_path):
    """Verifica che i file audio nel TSV esistano in questa sessione."""
    with open(tsv_path) as f:
        righe = f.readlines()
    if len(righe) < 2:
        return False
    esempio = righe[1].split('\t')[0].strip()
    return os.path.exists(esempio)

# Copia TSV da Drive se presenti
train_drv = f'{TSV_DIR_DRIVE}/train.tsv'
valid_drv = f'{TSV_DIR_DRIVE}/valid.tsv'

os.makedirs(TSV_DIR_LOCALE, exist_ok=True)

if os.path.exists(train_drv) and os.path.exists(valid_drv):
    shutil.copy(train_drv, TRAIN_TSV)
    shutil.copy(valid_drv, VALID_TSV)
    if tsv_audio_exists(TRAIN_TSV):
        print('TSV da Drive + file audio presenti — skip download')
    else:
        print('TSV da Drive ma file audio MANCANTI — rigenero...')
        os.remove(TRAIN_TSV)
        os.remove(VALID_TSV)

if not os.path.exists(TRAIN_TSV):
    print('Avvio prepare_fsd50k_tsv.py (~30-40 min la prima volta)...')
    !python {REPO_DIR}/scripts/prepare_fsd50k_tsv.py \
        --output_dir {TSV_DIR_LOCALE} \
        --audio_dir  {AUDIO_DIR}
    # Salva su Drive
    shutil.copy(TRAIN_TSV, train_drv)
    shutil.copy(VALID_TSV, valid_drv)
    print('TSV salvati su Drive')

# Verifica finale
for nome, path in [('train.tsv', TRAIN_TSV), ('valid.tsv', VALID_TSV)]:
    if os.path.exists(path):
        with open(path) as f:
            righe = f.readlines()
        prima = righe[0].strip()
        n_clip = len(righe) - 1
        print(f'OK {nome}: {n_clip} clip | prima riga: "{prima}"')
    else:
        print(f'MANCANTE {nome}')

In [ ]:
# CELLA 5 — Verifica setup completo
import os, torch

print('='*55)
print('  VERIFICA SETUP')
print('='*55)

checks = {
    'GPU disponibile':             torch.cuda.is_available(),
    'mae_ast/models/ presente':    os.path.exists(f'{USER_DIR}/models'),
    'mae_ast/criterions/ presente':os.path.exists(f'{USER_DIR}/criterions'),
    'mae_ast/tasks/ presente':     os.path.exists(f'{USER_DIR}/tasks'),
    'config/pretrain/ presente':   os.path.exists(CONFIG_DIR),
    'mae_ast.yaml presente':       os.path.exists(f'{CONFIG_DIR}/mae_ast.yaml'),
    'train.tsv presente':          os.path.exists(TRAIN_TSV),
    'valid.tsv presente':          os.path.exists(VALID_TSV),
    'Drive montato':               os.path.exists(CKPT_DIR_DRIVE),
}

tutti_ok = True
for desc, ok in checks.items():
    print(f'  {"OK" if ok else "MANCANTE"}  {desc}')
    if not ok:
        tutti_ok = False

print()
if tutti_ok:
    print('  Tutto pronto — esegui 6A, 6B, 6C in ordine')
else:
    print('  Risolvi i problemi prima di procedere')

print(f'\n  CONFIG_DIR: {CONFIG_DIR}')
print(f'  USER_DIR:   {USER_DIR}')
print(f'  DATA_DIR:   {TSV_DIR_LOCALE}')

In [ ]:
# CELLA 6A — Pretraining masking 25%
#
# Parametri verificati da config/pretrain/mae_ast.yaml:
#   - task._name: mae_ast_pretraining (calcola fbank internamente)
#   - model._name: mae_ast
#   - criterion._name: mae_ast
#   - optimization.clip_norm: 10.0 (dal config ufficiale)
#   - lr_scheduler.warmup_updates: 2000 (ridotto da 32000 per 20k steps)
#
# TEMPO STIMATO: ~50 min su A100

import os, time, glob, shutil

MASK     = '0.25'
RUN_NAME = 'patch_6L_mask025'
SAVE_DIR = f'/content/checkpoints/{RUN_NAME}'
os.makedirs(SAVE_DIR, exist_ok=True)

print(f'RUN: {RUN_NAME}  |  random_mask_prob={MASK}')
print(f'Task: mae_ast_pretraining (fbank calcolate internamente)')
print()

t0 = time.time()

# Comando dal README MAE-AST adattato per:
#   encoder_layers=6  (invece di 12, piu veloce)
#   max_update=20000  (invece di 400000, FSD50K e piu piccolo)
#   warmup_updates=2000 (ridotto proporzionalmente)
#   max_tokens=1400000  (dal config ufficiale)
os.system(f"""
export HYDRA_FULL_ERROR=1 && \
fairseq-hydra-train \
  --config-dir {CONFIG_DIR} \
  --config-name mae_ast \
  common.user_dir={USER_DIR} \
  task.data={TSV_DIR_LOCALE} \
  model.encoder_layers=6 \
  model.decoder_layers=2 \
  model.random_mask_prob={MASK} \
  task.mask_type=chunk_mask \
  model.ast_kernel_size_chan=16 \
  model.ast_kernel_size_time=16 \
  model.ast_kernel_stride_chan=16 \
  model.ast_kernel_stride_time=16 \
  criterion.classification_weight=1 \
  criterion.reconstruction_weight=10 \
  optimization.lr=[0.0001] \
  optimization.max_update=20000 \
  optimization.clip_norm=10.0 \
  dataset.max_tokens=1400000 \
  lr_scheduler.warmup_updates=2000 \
  lr_scheduler.total_num_update=20000 \
  distributed_training.distributed_world_size=1 \
  distributed_training.nprocs_per_node=1 \
  common.log_interval=100 \
  common.fp16=true \
  checkpoint.save_interval_updates=5000 \
  checkpoint.keep_interval_updates=1 \
  checkpoint.no_epoch_checkpoints=true \
  hydra.run.dir={SAVE_DIR}
""")

t1 = time.time()
print(f'\nRun 25% completata in {(t1-t0)/60:.1f} min')

# Copia checkpoint finale su Drive
ckpt_drive = f'{CKPT_DIR_DRIVE}/{RUN_NAME}'
os.makedirs(ckpt_drive, exist_ok=True)
checkpoints = sorted(glob.glob(f'{SAVE_DIR}/checkpoint_*.pt'))
if checkpoints:
    shutil.copy(checkpoints[-1], f'{ckpt_drive}/checkpoint_final.pt')
    print(f'Checkpoint -> Drive: {ckpt_drive}/checkpoint_final.pt')
else:
    # Cerca anche nella sottocartella che Hydra crea
    checkpoints = sorted(glob.glob(f'{SAVE_DIR}/**/checkpoint_*.pt', recursive=True))
    if checkpoints:
        shutil.copy(checkpoints[-1], f'{ckpt_drive}/checkpoint_final.pt')
        print(f'Checkpoint -> Drive: {ckpt_drive}/checkpoint_final.pt')
    else:
        print(f'ATTENZIONE: nessun checkpoint trovato in {SAVE_DIR}')
        print('Controlla i log sopra per errori')

In [ ]:
# CELLA 6B — Pretraining masking 50%
# Masking intermedio. TEMPO: ~35 min su A100

import os, time, glob, shutil

MASK     = '0.50'
RUN_NAME = 'patch_6L_mask050'
SAVE_DIR = f'/content/checkpoints/{RUN_NAME}'
os.makedirs(SAVE_DIR, exist_ok=True)

print(f'RUN: {RUN_NAME}  |  random_mask_prob={MASK}')
t0 = time.time()

os.system(f"""
export HYDRA_FULL_ERROR=1 && \
fairseq-hydra-train \
  --config-dir {CONFIG_DIR} \
  --config-name mae_ast \
  common.user_dir={USER_DIR} \
  task.data={TSV_DIR_LOCALE} \
  model.encoder_layers=6 \
  model.decoder_layers=2 \
  model.random_mask_prob={MASK} \
  task.mask_type=chunk_mask \
  model.ast_kernel_size_chan=16 \
  model.ast_kernel_size_time=16 \
  model.ast_kernel_stride_chan=16 \
  model.ast_kernel_stride_time=16 \
  criterion.classification_weight=1 \
  criterion.reconstruction_weight=10 \
  optimization.lr=[0.0001] \
  optimization.max_update=20000 \
  optimization.clip_norm=10.0 \
  dataset.max_tokens=1400000 \
  lr_scheduler.warmup_updates=2000 \
  lr_scheduler.total_num_update=20000 \
  distributed_training.distributed_world_size=1 \
  distributed_training.nprocs_per_node=1 \
  common.log_interval=100 \
  common.fp16=true \
  checkpoint.save_interval_updates=5000 \
  checkpoint.keep_interval_updates=1 \
  checkpoint.no_epoch_checkpoints=true \
  hydra.run.dir={SAVE_DIR}
""")

t1 = time.time()
print(f'\nRun 50% completata in {(t1-t0)/60:.1f} min')

ckpt_drive = f'{CKPT_DIR_DRIVE}/{RUN_NAME}'
os.makedirs(ckpt_drive, exist_ok=True)
checkpoints = sorted(glob.glob(f'{SAVE_DIR}/**/checkpoint_*.pt', recursive=True))
if checkpoints:
    shutil.copy(checkpoints[-1], f'{ckpt_drive}/checkpoint_final.pt')
    print(f'Checkpoint -> Drive: {ckpt_drive}/checkpoint_final.pt')

In [ ]:
# CELLA 6C — Pretraining masking 75%
# Run principale del paper. TEMPO: ~25 min su A100
# (piu veloce: encoder vede solo 25% dei token)

import os, time, glob, shutil

MASK     = '0.75'
RUN_NAME = 'patch_6L_mask075'
SAVE_DIR = f'/content/checkpoints/{RUN_NAME}'
os.makedirs(SAVE_DIR, exist_ok=True)

print(f'RUN: {RUN_NAME}  |  random_mask_prob={MASK}')
t0 = time.time()

os.system(f"""
export HYDRA_FULL_ERROR=1 && \
fairseq-hydra-train \
  --config-dir {CONFIG_DIR} \
  --config-name mae_ast \
  common.user_dir={USER_DIR} \
  task.data={TSV_DIR_LOCALE} \
  model.encoder_layers=6 \
  model.decoder_layers=2 \
  model.random_mask_prob={MASK} \
  task.mask_type=chunk_mask \
  model.ast_kernel_size_chan=16 \
  model.ast_kernel_size_time=16 \
  model.ast_kernel_stride_chan=16 \
  model.ast_kernel_stride_time=16 \
  criterion.classification_weight=1 \
  criterion.reconstruction_weight=10 \
  optimization.lr=[0.0001] \
  optimization.max_update=20000 \
  optimization.clip_norm=10.0 \
  dataset.max_tokens=1400000 \
  lr_scheduler.warmup_updates=2000 \
  lr_scheduler.total_num_update=20000 \
  distributed_training.distributed_world_size=1 \
  distributed_training.nprocs_per_node=1 \
  common.log_interval=100 \
  common.fp16=true \
  checkpoint.save_interval_updates=5000 \
  checkpoint.keep_interval_updates=1 \
  checkpoint.no_epoch_checkpoints=true \
  hydra.run.dir={SAVE_DIR}
""")

t1 = time.time()
print(f'\nRun 75% completata in {(t1-t0)/60:.1f} min')

ckpt_drive = f'{CKPT_DIR_DRIVE}/{RUN_NAME}'
os.makedirs(ckpt_drive, exist_ok=True)
checkpoints = sorted(glob.glob(f'{SAVE_DIR}/**/checkpoint_*.pt', recursive=True))
if checkpoints:
    shutil.copy(checkpoints[-1], f'{ckpt_drive}/checkpoint_final.pt')
    print(f'Checkpoint -> Drive: {ckpt_drive}/checkpoint_final.pt')

In [ ]:
# CELLA 7 — Misura speedup MAE-AST vs SSAST-style
#
# Replica empirica della Table 4 del paper.
# MAE-AST esclude i token mascherati dall'encoder —
# con mask 75% l'encoder processa solo 64 token invece di 256.
# La complessita dell'attention e O(n^2) quindi:
#   speedup_teorico = (256/64)^2 = 16x
# In pratica e ~3x per overhead fisso (embedding, norm, ecc.)

import torch, time, json, os

device = torch.device('cuda')
N_ITER = 200
BATCH  = 8
DIM    = 768

# Con spettrogramma 128x512 e patch 16x16:
# n_patch = (128/16) * (512/16) = 8 * 32 = 256 token totali
CONFIGS = [
    (256, 'SSAST-style (0% mask,  256 tok)'),
    (128, 'MAE-AST     (50% mask, 128 tok)'),
    (64,  'MAE-AST     (75% mask,  64 tok)'),
]

layer = torch.nn.TransformerEncoderLayer(
    d_model=DIM, nhead=12, dim_feedforward=3072,
    batch_first=True
).to(device).eval()

risultati = {}
for n_tok, desc in CONFIGS:
    x = torch.randn(BATCH, n_tok, DIM).to(device)
    # warmup GPU
    with torch.no_grad():
        for _ in range(20): _ = layer(x)
    torch.cuda.synchronize()
    t0 = time.time()
    with torch.no_grad():
        for _ in range(N_ITER): _ = layer(x)
    torch.cuda.synchronize()
    ms = (time.time() - t0) / N_ITER * 1000
    risultati[desc] = ms
    print(f'  {desc}: {ms:.2f} ms/step')

base = risultati['SSAST-style (0% mask,  256 tok)']
print()
for desc, ms in risultati.items():
    print(f'  Speedup {desc.split("(")[0].strip()}: {base/ms:.2f}x')
print(f'\n  Paper riporta: ~3x con mask 75%')

out = f'{DRIVE_DIR}/speedup.json'
with open(out, 'w') as f:
    json.dump({
        'ms_per_step': risultati,
        'speedup_50pct': base / risultati['MAE-AST     (50% mask, 128 tok)'],
        'speedup_75pct': base / risultati['MAE-AST     (75% mask,  64 tok)'],
        'paper_target_75pct': 3.0,
        'n_token_totali': 256,
        'gpu': torch.cuda.get_device_name(0),
    }, f, indent=2)
print(f'  Risultati salvati: {out}')

## Prossimi passi

Dopo che i tre checkpoint sono su Drive:
```
MyDrive/mae_ast_partA/checkpoints/
  patch_6L_mask025/checkpoint_final.pt
  patch_6L_mask050/checkpoint_final.pt
  patch_6L_mask075/checkpoint_final.pt
```

Esegui `PartA_finetune_ESC50.ipynb` che:
1. Converte i tre checkpoint fairseq -> timm
2. Fine-tuning 5-fold ESC-50 per ciascuno
3. Confronta le accuracy: verifica `acc_75 > acc_50 > acc_25`

La stessa limitazione della Parte B si applica qui:
il `patch_embed` non e nel checkpoint fairseq,
quindi viene inizializzato casualmente da timm.
Questo abbassa i numeri assoluti ma il **trend** tra masking ratio
dovrebbe essere comunque verificabile.